# Sionna OFDM grid channel estimation on Colab

Runs the Sionna resource-grid baselines and the small CNN comparison on a
Colab GPU. Set the runtime to GPU (`Runtime -> Change runtime type -> T4 GPU`)
before running.

## 1. GPU check

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

## 2. Clone the repository

In [ ]:
!git clone --branch main --single-branch https://github.com/s6ii5vii/sionna-neural-ofdm-channel-estimation.git
%cd sionna-neural-ofdm-channel-estimation

## 3. Install the ml stack

If the import cell below fails, run `Runtime -> Restart session` once, then
re-run from this install cell onward.

In [ ]:
!pip install -q --upgrade "numpy>=2.2.6,<2.3" "numba>=0.60,<0.62"
!pip install -q -e ".[ml,test]"

In [ ]:
import sionna, torch
print("sionna", sionna.__version__, "| torch", torch.__version__)
from sionna.phy import config
from sionna.phy.ofdm import ResourceGrid, ResourceGridMapper, LSChannelEstimator
from sionna.phy.channel import ApplyOFDMChannel, GenerateOFDMChannel
from sionna.phy.channel.tr38901 import TDL
from sionna.phy.mapping import QAMSource

## 4. Tests

In [ ]:
!python -m pytest -q

## 5. Least-squares grid baseline (LS-nn vs LS-lin)

In [ ]:
!python experiments/grid-tdl-v1/run-experiment.py

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv("results/tables/grid-tdl-v1.csv"))
display(Image("results/figures/grid-tdl-v1-nmse.png"))

## 6. Train the CNN and compare against LS

Training auto-generates the grid dataset if it does not already exist.

In [ ]:
!python -m channel_estimation.train experiments/grid-neural-comparison-v1/config.yaml

In [ ]:
import json
print(json.dumps(json.load(open("results/checkpoints/grid-neural-comparison-v1.report.json")), indent=2))

In [ ]:
!python experiments/grid-neural-comparison-v1/run-experiment.py

In [ ]:
display(pd.read_csv("results/tables/grid-neural-comparison-v1.csv"))
display(Image("results/figures/grid-neural-comparison-v1-nmse.png"))